In [1]:
import pandas as pd
import re

IN_PATH  = "../data/raw/articles_combined.csv"      # or your real combined filename
OUT_PATH = "../data/processed/articles_cleaned.csv" # recommended processed folder

df = pd.read_csv(IN_PATH)
print(f"Original size: {len(df)} rows")

# ----------------------------
# 1) Drop obvious bad rows
# ----------------------------
# Remove rows with scraping errors (keep only successful)
if "error" in df.columns:
    df = df[df["error"].isna() | (df["error"] == "")]

# Require url + content
df = df[df["url"].notna() & df["content"].notna()]
df["url"] = df["url"].astype(str)
df["content"] = df["content"].astype(str)

print(f"After removing errors/missing url/content: {len(df)} rows")

# ----------------------------
# 2) Normalize URLs (remove tracking params)
# ----------------------------
def clean_url(url: str) -> str:
    url = url.strip()

    # remove common tracking query params (utm_, ref, source, etc.)
    url = re.sub(r"(\?.*)$", "", url)  # drop all query string
    url = url.rstrip("/")

    # normalize dev.to: sometimes double slashes etc.
    url = re.sub(r"^http://", "https://", url)

    return url

df["url"] = df["url"].apply(clean_url)

# ----------------------------
# 3) Remove duplicates
# ----------------------------
df = df.drop_duplicates(subset=["url"], keep="first")
print(f"After removing duplicate URLs: {len(df)} rows")

# Sometimes titles repeat across sources; optional
df = df.drop_duplicates(subset=["title"], keep="first")
print(f"After removing duplicate titles: {len(df)} rows")

# ----------------------------
# 4) Clean text fields
# ----------------------------
df["title"] = df["title"].fillna("").astype(str).str.strip()
df["author"] = df["author"].fillna("Unknown").astype(str).str.strip()
df["tags"] = df["tags"].fillna("").astype(str).str.strip()
df["published_at"] = df["published_at"].fillna("")

# Create analysis text: title + content
df["full_text"] = (df["title"] + ". " + df["content"]).str.strip()

# Compute length
df["text_length"] = df["full_text"].str.len()

# ----------------------------
# 5) Filter too-short / low-quality pages
# ----------------------------
MIN_CHARS = 400  # adjust; for LDA, 200-600 is typical
df = df[df["text_length"] >= MIN_CHARS]
print(f"After filtering short docs (<{MIN_CHARS} chars): {len(df)} rows")

# Optional: remove "blocked" / bot pages by keyword
BLOCK_PATTERNS = [
    "verify you are", "captcha", "sign in", "access denied", "unusual traffic"
]
df = df[~df["full_text"].str.lower().apply(lambda t: any(p in t for p in BLOCK_PATTERNS))]
print(f"After filtering likely blocked pages: {len(df)} rows")

# ----------------------------
# 6) Save cleaned dataset
# ----------------------------
# Ensure output folder exists
import os
os.makedirs(os.path.dirname(OUT_PATH), exist_ok=True)

df.to_csv(OUT_PATH, index=False)
print(f"Saved cleaned dataset to: {OUT_PATH}")

# ----------------------------
# 7) Statistics
# ----------------------------
print("\n=== Cleaned Dataset Statistics ===")
print("Sources:", df["source"].value_counts().to_dict())

# tags are comma-separated; compute top tags
all_tags = (
    df["tags"].fillna("")
    .str.split(",")
    .explode()
    .str.strip()
)
all_tags = all_tags[all_tags != ""]
print("Top 15 tags:", all_tags.value_counts().head(15).to_dict())

print("Unknown authors:", int((df["author"].str.lower() == "unknown").sum()))


Original size: 1380 rows
After removing errors/missing url/content: 1380 rows
After removing duplicate URLs: 1142 rows
After removing duplicate titles: 1036 rows
After filtering short docs (<400 chars): 1026 rows
After filtering likely blocked pages: 1008 rows
Saved cleaned dataset to: ../data/processed/articles_cleaned.csv

=== Cleaned Dataset Statistics ===
Sources: {'dev.to': 469, 'techcrunch': 468, 'github_resources': 71}
Top 15 tags: {"['AI'": 185, '#ai': 117, '#webdev': 107, '#programming': 90, "'Startups'": 72, 'DevOps': 71, 'AI': 71, 'Security': 71, 'Software Development': 71, "'Apps'": 60, '#tutorial': 59, '#beginners': 49, '#javascript': 46, '#devops': 45, "'Startups']": 44}
Unknown authors: 469
